# 03 — BERTopic

BERTopic owns its own embedding (MiniLM) and clustering (UMAP + HDBSCAN
internally), so it's kept separate from notebook 02. Runs on the
**generated summary sentences**, forced to 16 topics
(`config.NUM_CLASSES`), with `KeyBERTInspired` topic representations for
readable labels.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import numpy as np
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

from utils import config
from utils.data import stratified_sample
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised, hungarian_match_predictions
from utils.samples import save_full_output

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
texts = train_sample["text"].tolist()
summaries = train_sample["summary"].tolist()
true_labels = train_sample["label"].to_numpy()

In [3]:
umap_model = UMAP(n_neighbors=15, n_components=5, metric="cosine", random_state=config.SEED)
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
vectorizer_model = CountVectorizer(stop_words="english", min_df=2)
representation_model = KeyBERTInspired()
topic_model = BERTopic(embedding_model=sentence_model, umap_model=umap_model,
                        vectorizer_model=vectorizer_model,
                        representation_model=representation_model,
                        nr_topics=config.NUM_CLASSES, calculate_probabilities=False)

topics, _ = topic_model.fit_transform(summaries)
print(topic_model.get_topic_info())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   Topic  Count                                   Name  \
0     -1    161            -1_biden_court_senate_trump   
1      0     32               0_covid_fda_shots_health   
2      1     20               1_earth_vr_energy_planet   
3      2     29    2_arrested_arresting_charges_police   
4      3     14            3_united_england_club_rugby   
5      4     20  4_students_university_student_schools   
6      5     43   5_trump_donald_weinstein_allegations   

                                      Representation  \
0  [biden, court, senate, trump, joe, president, ...   
1  [covid, fda, shots, health, virus, decades, tr...   
2  [earth, vr, energy, planet, obama, oil, resear...   
3  [arrested, arresting, charges, police, shootin...   
4  [united, england, club, rugby, league, france,...   
5  [students, university, student, schools, educa...   
6  [trump, donald, weinstein, allegations, ceo, f...   

                                 Representative_Docs  
0  [Bill de Blasio has asked th

In [4]:
topics_arr = np.array(topics)
topic_embeddings = sentence_model.encode(summaries, show_progress_bar=True, convert_to_numpy=True)
bertopic_metrics = evaluate_unsupervised(true_labels, topics_arr, topic_embeddings,
                                          metric_sample_size=config.SILHOUETTE_SAMPLE_SIZE, seed=config.SEED)
print(bertopic_metrics)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_bertopic.json", "w") as f:
    json.dump(bertopic_metrics, f, indent=2)
print("Saved BERTopic metrics.")

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

{'ACC (Hungarian)': np.float64(0.46835443037974683), 'Macro F1': 0.23356298887188448, 'NMI': 0.567647014032123, 'ARI': 0.33402618969874115, 'FMI': 0.44737182774768386, 'Homogeneity': 0.4713554262238026, 'Completeness': 0.7133812243114277, 'V-Measure': 0.5676470140321229, 'Silhouette Score': 0.08681689947843552, 'Davies-Bouldin': 3.648157594885076, 'Coverage': np.float64(0.4952978056426332)}
Saved BERTopic metrics.


### Topic inspection + full-row output

Same purity/majority-label crosstab and example-summary view used in
`02_unsupervised_clustering.ipynb`, plus a full-row
`text, summary, predicted_label, true_label` CSV over every document.

In [5]:
summary = summarize_clusters(summaries, topics_arr, true_labels, topic_embeddings, config.CLASS_NAMES)

if summary.empty:
    print("(no non-noise topics — everything was noise)")
else:
    with pd.option_context("display.max_colwidth", 60):
        print(summary.drop(columns="example_docs").to_string(index=False))
    for _, row in summary.iterrows():
        print(f"  topic {row['cluster']} examples:")
        for doc in row["example_docs"]:
            print(f"    - {doc}")
    summary.to_csv(config.RESULTS_DIR / "clusters_bertopic.csv", index=False)
    print(f"\nSaved per-topic qualitative summary to {config.RESULTS_DIR / 'clusters_bertopic.csv'}")

predicted = hungarian_match_predictions(true_labels, topics_arr)
save_full_output(
    texts, predicted, true_labels, config.CLASS_NAMES,
    extra_columns={"summary": summaries},
    path=config.RESULTS_DIR / "full_labels_bertopic.csv")
print("Saved full-row output for bertopic.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 cluster  size majority_true_label   purity                                                                                                                                                                                                                                        top_terms
       0    32              HEALTH 0.375000                           covid 19, continent zealandia discovered, zealandia discovered underwater, ban looms hunts, lake urmia iran, hunts held ban, shut canberra office, orangutans row boats, emergencies shut canberra, moderna urging fda
       1    20             SCIENCE 0.350000                       driving restrictions paris, mars brightest years, co2 generate electricity, astronomers hate dust, mars brightest, farmers testing 5g, testing 5g drones, opic support renewable, earth space vr, dollars cuba conversions
       2    29               CRIME 0.517241                  nabra hassanen killed, darren criss grieving, explosion fedex facility, brother char